# 3D Reconstruction: Depth Estimation & Point Cloud Fusion

Several related approaches to turning CanSat camera imagery into 3D terrain point clouds, from simplest to most complete.

## 1. Single-frame MiDaS depth → point cloud

Baseline approach: run MiDaS monocular depth estimation on one image and back-project it to 3D using a pinhole camera model.

In [ ]:
import os
import numpy as np
import open3d as o3d
import cv2
import torch

# โหลดโมเดล MiDaS
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
midas = torch.hub.load("isl-org/MiDaS", "DPT_Hybrid")
midas.to(device)
midas.eval()

# โหลด Transform
midas_transforms = torch.hub.load("isl-org/MiDaS", "transforms")
transform = midas_transforms.dpt_transform

# กำหนด path ของภาพ
image_paths = ["picture_test_ai9.jpg"] #[f"picture_test_ai{i}.jpg" for i in range(1, 11)]

point_clouds = []

for path in image_paths:
    img_raw = cv2.imread(path)
    if img_raw is None:
        print(f"⛔ โหลดไม่ได้: {path}")
        continue

    # ข้ามภาพที่สว่างเกินไป
    if np.mean(cv2.cvtColor(img_raw, cv2.COLOR_BGR2GRAY)) > 240:
        print(f"🚫 ข้ามภาพสว่างเกิน: {path}")
        continue

    # Resize เพื่อเร่งความเร็ว
    img_raw = cv2.resize(img_raw, (512, 288))
    img_rgb = cv2.cvtColor(img_raw, cv2.COLOR_BGR2RGB)
    input_tensor = transform(img_rgb).to(device)

    with torch.no_grad():
        prediction = midas(input_tensor)
        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1),
            size=img_rgb.shape[:2],
            mode="bilinear",
            align_corners=False,
        ).squeeze()

    depth = prediction.cpu().numpy()

    if np.count_nonzero(depth) < 100:
        print(f"⚠️ Depth บางเกิน: {path}")
        continue

    h, w = depth.shape
    fx = fy = w / 2
    cx = w / 2
    cy = h / 2

    points, colors = [], []

    for y in range(h):
        for x in range(w):
            z = depth[y, x]
            if z <= 0:
                continue
            X = (x - cx) * z / fx
            Y = (y - cy) * z / fy
            points.append([X, Y, z])
            colors.append(img_rgb[y, x] / 255.0)

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(np.array(points))
    pcd.colors = o3d.utility.Vector3dVector(np.array(colors))
    point_clouds.append(pcd)

# รวม Point Cloud
if point_clouds:
    merged_pcd = point_clouds[0]
    for pcd in point_clouds[1:]:
        merged_pcd += pcd

    o3d.io.write_point_cloud("test_3d_field_model.ply", merged_pcd)
    print("✅ บันทึก: 3d_field_model.ply")
    o3d.visualization.draw_geometries([merged_pcd])
else:
    print("❌ ไม่มี point cloud ที่ใช้ได้")


## 2. Multi-frame reconstruction using IMU pose (roll/pitch)

Extends the single-frame approach across a sequence of images, using logged IMU roll/pitch (from `cam-mpu/aligned_pose.csv`) to rotate each frame's point cloud into a shared world frame before merging.

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
import open3d as o3d
import math

# === Load MiDaS ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
midas = torch.hub.load("isl-org/MiDaS", "DPT_Hybrid")
midas.to(device).eval()
midas_transforms = torch.hub.load("isl-org/MiDaS", "transforms")
transform = midas_transforms.dpt_transform

# === Load Pose CSV (image + roll + pitch only) ===
df = pd.read_csv("cam-mpu\\aligned_pose.csv", nrows=18)  # Must have columns: image, roll, pitch

#drop first 5 rows
df = df.drop(df.index[:5])

# === Rotation helper ===
def euler_to_rotmat(roll, pitch, yaw=0):
    roll, pitch, yaw = np.radians([roll, pitch, yaw])
    Rx = np.array([[1, 0, 0],
                   [0, np.cos(roll), -np.sin(roll)],
                   [0, np.sin(roll), np.cos(roll)]])
    Ry = np.array([[np.cos(pitch), 0, np.sin(pitch)],
                   [0, 1, 0],
                   [-np.sin(pitch), 0, np.cos(pitch)]])
    Rz = np.array([[np.cos(yaw), -np.sin(yaw), 0],
                   [np.sin(yaw), np.cos(yaw), 0],
                   [0, 0, 1]])
    return Rz @ Ry @ Rx

# === Reconstruct World Point Cloud ===
merged_points, merged_colors = [], []

for _, row in df.iterrows():
    img_path = "cam-mpu\\" + row['image']
    img_raw = cv2.imread(img_path)
    if img_raw is None:
        print(f"⛔ Image not found: {img_path}")
        continue

    # ตรวจสอบว่าไม่ใช่ภาพขาวเกินไป
    img_gray = cv2.cvtColor(img_raw, cv2.COLOR_BGR2GRAY)
    if np.mean(img_gray) > 240:
        print(f"ข้ามภาพ (สว่างเกินไป): {img_path}")
        continue

    # Resize เพื่อลดภาระการประมวลผล (optional)
    img_raw = cv2.resize(img_raw, (384, 256))  # หรือ (512, 288)

    img = cv2.cvtColor(img_raw, cv2.COLOR_BGR2RGB)
    input_tensor = transform(img).to(device)


    with torch.no_grad():
        depth = midas(input_tensor)
        depth = torch.nn.functional.interpolate(
            depth.unsqueeze(1),
            size=img.shape[:2],
            mode="bilinear",
            align_corners=False
        ).squeeze().cpu().numpy()

    h, w = depth.shape
    fx = fy = w / 2
    cx, cy = w / 2, h / 2

    # Pose from IMU only
    t = np.array([0, 0, 0])
    R = euler_to_rotmat(row['roll'], row['pitch'], math.atan2(row['roll'],row['pitch']) )  # Assume yaw = formula

    for v in range(0, h):
        for u in range(0, w):
            d = depth[v, u]
            if d <= 0: continue
            X = (u - cx) * d / fx
            Y = (v - cy) * d / fy
            pt_cam = np.array([X, Y, d])
            pt_world = R @ pt_cam + t
            merged_points.append(pt_world)
            merged_colors.append(img[v, u] / 255.0)

# === Delete null points ===
merged_points = np.array(merged_points)
merged_colors = np.array(merged_colors)

# NaN / Inf
mask = np.all(np.isfinite(merged_points), axis=1) & (merged_points.shape[1] == 3)
merged_points = merged_points[mask]
merged_colors = merged_colors[mask]


# === Save Point Cloud ===
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(merged_points)
pcd.colors = o3d.utility.Vector3dVector(merged_colors)
o3d.io.write_point_cloud("midas_mpu_model_with_imu.ply", pcd)
print("✅ Saved: midas_mpu_model_with_imu.ply")


## View a saved point cloud

In [ ]:
import open3d as o3d

# Load .ply file
#pcd = o3d.io.read_point_cloud("midas_1_cam_test2.ply")
pcd = o3d.io.read_point_cloud("midas_left_n_right.ply")

print(pcd)
print("Number of points:", len(pcd.points))

o3d.visualization.draw_geometries([pcd])


## 3. Stereo disparity (SGBM) → point cloud

An alternative to monocular depth: compute disparity between a stereo image pair directly and reproject to 3D, saving both a `.ply` and a flat `.txt` point list.

In [ ]:
import cv2
import numpy as np
import pandas as pd

# --- CONFIG --- #
image_path_left = r"combine-2\caml\test2\picture_cam_l20_s.jpg"
image_path_right = r"combine-2\camr\test2\picture_cam_r20_s.jpg"
fx = 850.0
baseline = 0.218  # meters
width, height = 1600, 1200

cx = width / 2
cy = height / 2
K = np.array([[fx, 0, cx],
              [0, fx, cy],
              [0, 0, 1]], dtype=np.float32)

# --- Load images --- #
imgL = cv2.imread(image_path_left, cv2.IMREAD_GRAYSCALE)
imgR = cv2.imread(image_path_right, cv2.IMREAD_GRAYSCALE)
colorL = cv2.imread(image_path_left, cv2.IMREAD_COLOR)

imgL = cv2.resize(imgL, (width, height))
imgR = cv2.resize(imgR, (width, height))
colorL = cv2.resize(colorL, (width, height))

# --- Stereo Matching --- #
stereo = cv2.StereoSGBM_create(
    minDisparity=0,
    numDisparities=128,
    blockSize=9,
    P1=8 * 3 * 9 ** 2,
    P2=32 * 3 * 9 ** 2,
    disp12MaxDiff=1,
    uniquenessRatio=10,
    speckleWindowSize=100,
    speckleRange=32,
    preFilterCap=63,
    mode=cv2.STEREO_SGBM_MODE_SGBM_3WAY
)
disparity = stereo.compute(imgL, imgR).astype(np.float32) / 16.0
disparity[disparity <= 0] = 0.1

# --- Generate keypoints from disparity --- #
matches = []
step = 10
for v in range(0, height, step):
    for u in range(0, width, step):
        d = disparity[v, u]
        if d > 0 and u - d > 0:
            ptL = (u, v)
            ptR = (int(u - d), v)
            matches.append((ptL, ptR))

# --- Estimate depth map --- #
depth_map = fx * baseline / disparity

# --- Convert matched keypoints to 3D (left image) --- #
def pixel_to_3d(u, v, depth, K):
    z = depth[v, u]
    x = (u - K[0, 2]) * z / K[0, 0]
    y = (v - K[1, 2]) * z / K[1, 1]
    return [x, y, z]

points_3d = []
colors = []

for (u, v), _ in matches:
    pt3d = pixel_to_3d(u, v, depth_map, K)
    rgb = colorL[v, u]
    points_3d.append(pt3d)
    colors.append(rgb)

# --- Save as .txt --- #
points_3d = np.array(points_3d)
colors = np.array(colors)
df = pd.DataFrame(np.hstack((points_3d, colors)), columns=["x", "y", "z", "r", "g", "b"])
txt_path = "pointcloud_stereo_autoalign.txt"
df.to_csv(txt_path, index=False)

## Merge two point clouds via ICP registration

Aligns and merges two previously-saved point clouds with iterative closest point (ICP) registration.

In [ ]:
import open3d as o3d

# Load both point clouds
pcd1 = o3d.io.read_point_cloud("midas_1_cam_test2.ply")
pcd2 = o3d.io.read_point_cloud("update-ai\stereo-vision-master\data\output\test_point_cloud_250.ply")

# Rough alignment first (if needed)
trans_init = np.eye(4)

# Refine alignment using ICP
reg = o3d.pipelines.registration.registration_icp(
    source=pcd2,
    target=pcd1,
    max_correspondence_distance=0.05,
    init=trans_init,
    estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint()
)

# Apply transformation
pcd2.transform(reg.transformation)

# Combine
combined = pcd1 + pcd2
o3d.io.write_point_cloud("merged_output_aligned.ply", combined)


## Load exported stereo points from text and save as `.ply`

In [ ]:
import open3d as o3d
import numpy as np
import os

# Load data
point_cloud_data = np.loadtxt('pointcloud_stereo_autoalign.txt',skiprows=1, delimiter=',')  # replace with your data source

# Create Open3D PointCloud
point_cloud = o3d.geometry.PointCloud()
object_points = point_cloud_data[:, :3]
point_cloud.points = o3d.utility.Vector3dVector(object_points)

# Assign colors (assume columns 3 to 5 are R, G, B)
point_cloud.colors = o3d.utility.Vector3dVector(point_cloud_data[:, 3:6] / 255.0)

# Visualize
o3d.visualization.draw_geometries([point_cloud], window_name='Point Cloud Visualization', width=800, height=600)

# Save
os.makedirs('./data/output', exist_ok=True)
o3d.io.write_point_cloud('./data/output/test_point_cloud_250.ply', point_cloud)

## 4. Combined stereo + MiDaS fusion pipeline

The most complete approach: warps the right stereo image onto the left using the disparity map, blends the pair, then runs MiDaS depth estimation on the fused image.

In [ ]:
# Stereo + MiDaS 3D Point Cloud Fusion Pipeline

import cv2
import numpy as np
import torch
import urllib.request
from PIL import Image
import matplotlib.pyplot as plt
import open3d as o3d
from midas.dpt_depth import DPTDepthModel
from torchvision.transforms import Compose, Resize, ToTensor, Normalize

# --- Step 1: Load Stereo Images ---
imgL_path = r"combine-2\caml\test2\picture_cam_l20_s.jpg"
imgR_path = r"combine-2\camr\test2\picture_cam_r20_s.jpg"

imgL = cv2.imread(imgL_path)
imgR = cv2.imread(imgR_path)

imgL_gray = cv2.cvtColor(imgL, cv2.COLOR_BGR2GRAY)
imgR_gray = cv2.cvtColor(imgR, cv2.COLOR_BGR2GRAY)

# Resize for speed
image_width, image_height = 1600, 1200
imgL = cv2.resize(imgL, (image_width, image_height))
imgR = cv2.resize(imgR, (image_width, image_height))
imgL_gray = cv2.resize(imgL_gray, (image_width, image_height))
imgR_gray = cv2.resize(imgR_gray, (image_width, image_height))

# --- Step 2: Compute Disparity Map ---
stereo = cv2.StereoSGBM_create(
    minDisparity=0,
    numDisparities=128,
    blockSize=9,
    P1=8 * 3 * 9 ** 2,
    P2=32 * 3 * 9 ** 2,
    disp12MaxDiff=1,
    uniquenessRatio=10,
    speckleWindowSize=100,
    speckleRange=32,
    preFilterCap=63,
    mode=cv2.STEREO_SGBM_MODE_SGBM_3WAY
)
disparity = stereo.compute(imgL_gray, imgR_gray).astype(np.float32) / 16.0

disparity[disparity <= 0] = 0.1

# --- Step 3: Align Right Image to Left (Disparity Warping) ---
def warp_image(img, disparity):
    h, w = disparity.shape
    map_x = np.tile(np.arange(w), (h, 1)) - disparity
    map_y = np.tile(np.arange(h).reshape(-1, 1), (1, w))
    map_x = map_x.astype(np.float32)
    map_y = map_y.astype(np.float32)
    return cv2.remap(img, map_x, map_y, interpolation=cv2.INTER_LINEAR)

alignedR = warp_image(imgR, disparity)
fused_img = cv2.addWeighted(imgL, 0.5, alignedR, 0.5, 0)

# --- Step 4: Apply MiDaS Depth Estimation ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_type = "DPT_Large"

# Load MiDaS model
midas = torch.hub.load("isl-org/MiDaS", model_type)
midas.to(device).eval()

midas_transforms = torch.hub.load("isl-org/MiDaS", "transforms")
transform = midas_transforms.dpt_transform

img = Image.fromarray(cv2.cvtColor(fused_img, cv2.COLOR_BGR2RGB))
img.save("fused_image.jpg")  # or .png

